This is meant to account for the energetic and proteome costs (preliminary estimate) of various processes in the E matrix relative to M and S processes.


In [4]:
import pandas as pd

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *


load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [6]:
# proteomics dataset from human T cell atlas: https://tcellatlas.kaust.edu.sa/

cd4_prot = pd.read_csv(local_data_path + 'raw/CD4_human_TCA.csv')
# get average expression across 3 donors
cd4_prot = cd4_prot.groupby(['Protein ID', 'Compartment', 'TCR Stimulation (min)'])['Abundance'].mean().reset_index()
cd4_prot = cd4_prot.groupby(['Protein ID', 'Compartment'])['Abundance'].mean().reset_index()
cd4_tot = cd4_prot['Abundance'].sum()

In [7]:
# initialize cost DF
myindex = pd.MultiIndex(levels=[[],[],[], []],
                            codes=[[],[],[], []], # labels=[[],[],[], []],
                             names=['Process', 'Subprocess', 'Substrate', 'Reaction'])
cost = pd.DataFrame(index = myindex, 
                    columns = ['Energetic Cost', 'Energetic Unit', 'Machinery Abundance', 
                               'Required Machinery (HUGO Symbol)', 'Required Machinery (HUGO ID)',
                               'Compartment',
                               'Energetic Citation', 'Required Machinery Citation', 'Comments'])

From Jahir’s ATP cost analysis on my 676 secPs list:

1) Aa + translation average ATP cost is 2480

2) Secretion average ATP cost is 36


# Basic Calculations

In [8]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5199132/
L_premrna = 67e3
n_intron = 10
L_intron = 6355
n_exon = 11
L_exon = 309


L_polyA_n = 250 # https://www.nature.com/articles/s41592-019-0503-y
L_polyA_c = 100 # https://www.nature.com/articles/s41592-019-0503-y
# assume, on average, half the exons are included
# this matches well with average number of exons included 
# https://bionumbers.hms.harvard.edu/bionumber.aspx?id=101461&ver=2&trm=exons+included&org=
L_mrna = L_premrna - (n_intron*L_intron) - (0.5*n_exon*L_exon) + L_polyA_c

L3_utr = 520 # https://bionumbers.hms.harvard.edu/bionumber.aspx?id=103818&ver=2&trm=length+premrna&org=
L5_utr = 150 # https://bionumbers.hms.harvard.edu/bionumber.aspx?id=103819&ver=2&trm=length+premrna&org=

# half exon assumption gives a protein length that matches well with reported, e.g., 
# https://bionumbers.hms.harvard.edu/bionumber.aspx?id=106445&ver=4&trm=protein+length&org=
L_protein = (L_mrna - L3_utr - L5_utr)/3 

# Transcription

In [9]:
# nucleotide synthesis
energetic_cost = 46*L_premrna
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']

cost.loc[('Expression', 
          'Transcription', 
          'premRNA', 
          'rNTP_synthesis'),:] = [energetic_cost, 'ATP per molecule'] + [float('nan')]*4 + [eng_cit]+ [float('nan')]*2


In [10]:
#transcription initiation
# machinery database

rnap2 = pd.read_csv(local_data_path + 'raw/RNAP2_HUGO.csv', index_col = None, skiprows = [0])
gtf_1 = pd.read_csv(local_data_path + 'raw/pic_gtf_1.csv', index_col = None, skiprows = [0])
gtf_2 = pd.read_csv(local_data_path + 'raw/pic_gtf_2.csv', index_col = None, skiprows = [0])

gtf = pd.concat([gtf_1, gtf_2], axis = 0)
gtf.reset_index(inplace = True, drop = True)
gtf.drop_duplicates(keep = 'first', subset = ['Approved symbol'], inplace = True)

mediator = pd.read_csv(local_data_path + 'raw/mediator_complex.csv', index_col = None, skiprows = [0])

initiation_cost, activation_cost = 50, 10
energetic_cost = initiation_cost + activation_cost
unit = 'ATP per molecule'
machinery_symb = {'RNAP2': rnap2.loc[:, 'Approved symbol'].tolist(), 
                 'General TFs': gtf.loc[:, 'Approved symbol'].tolist(), 
                 'Mediator Complex': mediator.loc[:, 'Approved symbol'].tolist()}
machinery_id = {'RNAP2': rnap2.loc[:, 'HGNC ID (gene)'].tolist(), 
                'General TFs': gtf.loc[:, 'HGNC ID (gene)'].tolist(),
               'Mediator Complex': mediator.loc[:, 'HGNC ID (gene)'].tolist()}
machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot


compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3612373/']
comment = 'ATP Cost is associated with TFIIH helicase/translocase activity to form an open complex.'
comment += 'Includes transcription activation and initiation from supp_table_3. Machinery is just an'
comment += 'aggregation of gene groups from HGNC database.'

cost.loc[('Expression', 
          'Transcription', 
          'premRNA', 
          'Transcription_Initiation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [11]:
# capping: pausing transition b/w initiation and effective elongation

energetic_cost = 4
unit = 'ATP per molecule'
machinery_symb = {'General': ['RNGTT', 'RNMT','CMTR1','CMTR2','PCIF1','NCBP1','CDK7','SSU72','CDK9',
                              'CCNT1','SUPT4H1','SUPT5H','PAIP2B', 'PAIP2', 'NCBP2'],
                  'NELF':['NELFA', 'NELFB', 'NELFCD', 'NELFE']}
machinery_id = {'General': ['HGNC:10073', 'HGNC:10075', 'HGNC:21077', 'HGNC:25635', 'HGNC:16200', 
                           'HGNC:7658', 'HGNC:1778', 'HGNC:25016', 'HGNC:1780', 'HGNC:1599', 'HGNC:11467', 
                           'HGNC:11469', 'HGNC:29200', 'HGNC:17970', 'HGNC:7659'], 
                'NELF': ['HGNC:12768', 'HGNC:24324', 'HGNC:15934', 'HGNC:13974']}
machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://portlandpress.com/bioscirep/article/40/1/BSR20192825/221784/Interplay-of-mRNA-capping-and-transcription']
comment = 'Table 1 of source is the list of machinery'

cost.loc[('Expression', 
          'Transcription', 
          'premRNA', 
          'Capping'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [12]:
# CTD Phosphorylation: allows transcirption elongation
ptefb = pd.read_csv(local_data_path +'raw/ptefb.csv', index_col = None, skiprows = [0])


energetic_cost = 100
unit = 'ATP per molecule'
machinery_symb = {'P-TEFb': ptefb.loc[:, 'Approved symbol'].tolist()}
machinery_id = {'P-TEFb': ptefb.loc[:, 'HGNC ID (gene)'].tolist()}
machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3465734/']
comment = 'Machinery list from HGNC "P-TEFb complex subunits'

cost.loc[('Expression', 
          'Transcription', 
          'RNAP2', 
          'CTD_Phosphorylation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [13]:
# Transcription Elongation
elongin = pd.read_csv(local_data_path + 'raw/elongin.csv', index_col = None, skiprows = [0])
elongator = pd.read_csv(local_data_path + 'raw/elongator.csv', index_col = None, skiprows = [0])


n_histones = 0.0056*L_premrna
histone_modification_cost = (n_histones/8)*30
elongation_cost = 1
energetic_cost = histone_modification_cost + elongation_cost
unit = 'ATP per molecule'
machinery_symb = {'RNAP2': rnap2.loc[:, 'Approved symbol'].tolist(), 
                 'General': ['TCEA1'],
                 'Elongation Factors 1': ['GTF2F1', 'GTF2F2', 'ELL', 'ELL2', 'ELL3', 
                                         ], 
                 'Elongation Factors 2 - Elongin': elongin.loc[:, 'Approved symbol'].tolist(), 
                 'Nucleosome Modification': ['SUPT16H'] + elongator.loc[:, 'Approved symbol'].tolist()}
machinery_id = {'RNAP2': rnap2.loc[:, 'HGNC ID (gene)'].tolist(),  
               'General': ['HGNC:11612'], 
               'Elongation Factors 1': ['HGNC:4652', 'HGNC:4653', 'HGNC:23114', 'HGNC:17064', 'HGNC:23113'], 
               'Elongation Factors 2 - Elongin': elongin.loc[:, 'HGNC ID (gene)'].tolist(), 
               'Nucleosome Modification': ['HGNC:11465'] + elongator.loc[:, 'HGNC ID (gene)'].tolist()}
machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.annualreviews.org/doi/10.1146/annurev.biochem.72.121801.161551', 
                'https://bionumbers.hms.harvard.edu/files/Eukaryotic%20elongation%20factors.pdf']
comment = 'Includes histone modification costs, which may not be included in model'

cost.loc[('Expression', 
          'Transcription', 
          'preMRNA', 
          'Transcription_Elongation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [35]:
cost.iloc[4,]['Required Machinery Citation']

['https://www.annualreviews.org/doi/10.1146/annurev.biochem.72.121801.161551',
 'https://bionumbers.hms.harvard.edu/files/Eukaryotic%20elongation%20factors.pdf']

In [14]:
# polyA tail and transcription termination<--the two processes are coupled


rATP_cost = 49.7
rNTP_synthesis_cost = L_polyA_n*rATP_cost # 1 ATP synthesized per rAMP added
polyA_ligation_cost = L_polyA_n*1 #1 ATP consumed per rAMP added
energetic_cost = 'rNTP synthesis: ' + str(rNTP_synthesis_cost) + '\n polyA ligation: ' + str(polyA_ligation_cost)
unit = 'ATP per molecule'

machinery_symb = {'CPSF Complex': ['CPSF1', 'CPSF4', 'FIP1L1', 'CPSF2', 'CPSF3', 'WDR33', 'CPSF6'],
                  'CSTF': ['CSTF1', 'CSTF2', 'CSTF3'],
                 'polyA polymerase': ['PAPOLA', 'PAPOLB', 'PAPOLG'], 
                 'CFIm': ['NUDT21', 'CPSF7', 'CPSF6'], 
                 'CFIIm': ['PCF11', 'CLP1']}
machinery_id = {'CPSF Complex': ['HGNC:2324', 'HGNC:2327', 'HGNC:19124', 'HGNC:2325', 'HGNC:2326', 'HGNC:25651', 
                                'HGNC:13871'], 
               'CSTF': ['HGNC:2483', 'HGNC:2484', 'HGNC:2485'], 
               'polyA polymerase': ['HGNC:14981', 'HGNC:15970', 'HGNC:14982'], 
               'CFIm': ['HGNC:13870', 'HGNC:30098', 'HGNC:13871'], 
               'CFIIm': ['HGNC:30097', 'HGNC:16999']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://elifesciences.org/articles/33111', 'http://genesdev.cshlp.org/content/29/9/889.full', 
                'http://genesdev.cshlp.org/content/23/11/1247.full', 'https://pubmed.ncbi.nlm.nih.gov/9659921/']
comment = 'polyA synthesis and transcription termination are tightly coupled'

cost.loc[('Expression', 
          'Transcription', 
          'preMRNA', 
          'polyA_synthesis_transcription_termination'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [15]:
# RNAP dephosphorylation 

energetic_cost = float('nan')
unit = 'ATP per molecule'

# hgnc: https://www.genenames.org/data/genegroup/#!/group/1041
ctd_p = pd.read_csv(local_data_path + 'raw/ctd_phosphotase.csv', index_col = None, skiprows = [0])

machinery_symb = {'General': ctd_p.loc[:, 'Approved symbol'].tolist()}
machinery_id = {'General': ctd_p.loc[:, 'HGNC ID (gene)'].tolist()}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = []
machinery_cit = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4761483/'] 
comment = ''

cost.loc[('Expression', 
          'Transcription', 
          'RNAP', 
          'CTD_dephosphorylation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [16]:
# splicing 

cost_per_intron = 10
avg_cost_per_preMRNA = cost_per_intron*n_intron
energetic_cost = avg_cost_per_preMRNA
unit = 'ATP per molecule'

spliceosome = pd.read_csv(local_data_path + 'raw/spliceosome.txt', index_col = None, sep = '\t')
machinery_symb = {'Spliceosome': spliceosome.loc[:, 'Approved symbol'].tolist()}
machinery_id = {'Spliceosome': spliceosome.loc[:, 'HGNC ID'].tolist()}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:29322'] # just used HGNC
comment = 'Reports of splicing and export being coupled, see https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2265164/'

cost.loc[('Expression', 
          'Transcription', 
          'preMRNA', 
          'splicing'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [17]:
# lariat degradation 

energetic_cost = 'Helicase activity'
unit = 'ATP per molecule'


exosome = pd.read_csv(local_data_path + 'raw/exosome.csv', index_col = None, skiprows = [0])


machinery_symb = {'Linearization': ['DBR1'], 
                 "5' Degradation": ['XRN2'], 
                 "Exosome": exosome.loc[:, 'Approved symbol'].tolist() + ['C1D'], 
                 'NEXT': ['MTREX', 'RBM7', 'ZCCHC8']}
machinery_id = {'Linearization': ['HGNC:15594'], 
                 "5' Degradation": ['HGNC:12836'], 
               "Exosome": exosome.loc[:, 'HGNC ID (gene)'].tolist() + ['HGNC:29911'], 
               'NEXT': ['HGNC:18734', 'HGNC:9904', 'HGNC:25265']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus'
eng_cit = []
machinery_cit = ['https://www.sciencedirect.com/science/article/pii/S0076687901425481?via%3Dihub', 
                'https://link.springer.com/article/10.1007/s00438-011-0635-y#Sec1', 
                'https://www.sciencedirect.com/science/article/pii/S0092867409000671#fig1', 
                'https://www.cell.com/molecular-cell/fulltext/S1097-2765(11)00572-7?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS1097276511005727%3Fshowall%3Dtrue'] 
comment = 'Assume all lariats are efficiently degraded in the nucleus. This should be included because it will'
comment += ' reduce the cost associated with rNTP synthesis via recycling'

cost.loc[('Expression', 
          'Transcription', 
          'intron', 
          'lariat degradation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [18]:
# export 

energetic_cost = 10
unit = 'ATP per molecule'

tho = pd.read_csv(local_data_path + 'raw/tho.csv', index_col = None, skiprows = [0])

machinery_symb = {'TREX': tho.loc[:, 'Approved symbol'].tolist() + ['DDX39B', ]}
machinery_id = {'TREX': tho.loc[:, 'HGNC ID (gene)'].tolist() + ['HGNC:13917']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'nucleus to cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1172058/',
                 'https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0010144'
                 'https://jbiol.biomedcentral.com/articles/10.1186/jbiol217'] 
comment = 'TREX-dependent export is coupled to splicing, and only works for spliced mRNA. While '
comment += 'https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0010144 suggests '
comment += '3 independent pathways, we will assume only the TREX-dependent pathway.'

cost.loc[('Expression', 
          'Transcription', 
          'mRNA', 
          'export'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [19]:
# degradation

# Step 1: Deadenylation
energetic_cost = float('nan')
unit = 'ATP per molecule'


ccr4_not = pd.read_csv(local_data_path + 'raw/CCR4_NOT.csv', index_col = None, skiprows = [0])

# first step is deadenylation, either of the two complexes can act on it
machinery_symb = {'CCR4_NOT Deadenylation': ccr4_not.loc[:, 'Approved symbol'].tolist(), 
                  'PARN Deadenylation': ['PARN'], 
                 'PABP Deadenylation': ['PAN2', 'PAN3']}
machinery_id = {'CCR4_NOT Deadenylation': ccr4_not.loc[:, 'HGNC ID (gene)'].tolist(), 
                'PARN Deadenylation': ['HGNC:8609'], 
               'PABP Deadenylation': ['HGNC:20074', 'HGNC:29991']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = []
machinery_cit = ['https://www.nature.com/articles/nrm2104?proof=true', 
                'https://pubmed.ncbi.nlm.nih.gov/14583602/'] 
comment = "Step 1 in degradation is deadenylation. Any of the complexes can do this (OR rule)."
cost.loc[('Expression', 
          'Transcription', 
          'mRNA', 
          "deadenylation"),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]





# Pathway 1: 3'-->5'
energetic_cost = 'Helicase will make this a function of length'
unit = 'ATP per molecule'

machinery_symb = {"Exosome": exosome.loc[:, 'Approved symbol'].tolist() + ['C1D'], 
                 'Cap_Degradation': ['DCPS']}
machinery_id = {"Exosome": exosome.loc[:, 'HGNC ID (gene)'].tolist() + ['HGNC:29911'], 
               'Cap_Degradation': ['HGNC:29812']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = []
machinery_cit = ['https://www.nature.com/articles/nrm2104?proof=true'] 
comment = 'Only deadenylated polyA will be degraded'
comment += ' . Degradation energetic costs, while note included, involve helicase ATP hydrolysis and are '
comment += 'likely substantial'


cost.loc[('Expression', 
          'Transcription', 
          'mRNA', 
          "3'->5'_degradation"),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]


# Pathway 2: 5'-->3'
energetic_cost = ''
unit = 'ATP per molecule'

machinery_symb = {'LSM1-7 Complex': ['LSM1', 'LSM2', 'LSM3', 'LSM4', 'LSM5', 'LSM6', 'LSM7'] , 
                  'Decapping': ['DCP1A', 'DCP1B', 'DCP2'], 
                 "5' Exonuclease": ['XRN1']}
machinery_id = {'LSM1-7 Complex': ['HGNC:20472', 'HGNC:13940', 'HGNC:17874', 'HGNC:17259', 
                                   'HGNC:17162', 'HGNC:17017', 'HGNC:20470'],
                'Decapping': ['HGNC:18714', 'HGNC:24451', 'HGNC:24452'], 
               "5' Exonuclease": ['HGNC:30654']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = []
machinery_cit = ['https://www.nature.com/articles/nrm2104?proof=true'] 
comment = 'Only deadenylated polyA will be degraded'


cost.loc[('Expression', 
          'Transcription', 
          'mRNA', 
          "decapping_degradation"),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

# Translation

In [20]:
# # template
# # translation initiation

# energetic_cost = 
# unit = 'ATP per molecule'

# tho = pd.read_csv(local_data_path + 'raw/tho.csv', index_col = None, skiprows = [0])
# machinery_symb = {'': tho.loc[:, 'Approved symbol'].tolist()}
# machinery_id = {'': tho.loc[:, 'HGNC ID (gene)'].tolist()}

# machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
# machinery_cost = machinery_cost/cd4_tot

# compartment = 'cytoplasm'
# eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
# machinery_cit = [''] 
# comment = ''

# cost.loc[('Expression', 
#           'Translation', 
#           'mRNA', 
#           'initiation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
#                                             compartment, eng_cit, machinery_cit, comment]

In [21]:
# aa synth

average_aa_synth = 20 # rough estimate from table
energetic_cost = L_protein*average_aa_synth
unit = 'ATP per molecule'

machinery_symb = {}
machinery_id = {}

compartment = 'cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = [''] 
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'aa', 
          'aa_synthesis'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [22]:
# translation initiation

# 48S complex formation
energetic_cost = '10 - Helicase activity, once per multiple translations (circular mRNA)'
unit = 'ATP per n molecules'

eif2_complex, eif2_hgnc = ['EIF2S1', 'EIF2S2', 'EIF2S3'], ['HGNC:3265', 'HGNC:3266', 'HGNC:3267']
ternary_complex, ternary_hgnc = eif2_complex + ['Met-tRNA', 'GTP'], eif2_hgnc + [float('nan'), float('nan')]
eif3_complex = pd.read_csv(local_data_path + 'raw/eif3.csv', index_col = None, skiprows = [0])
forty_s_r = ['RPSA', 'RPS2', 'RPS3', 'RPS3A', 'RPS4X', 'RPS4Y', 'RPS5', 'RPS6', 'RPS7', 'RPS8', 'RPS9', 'RPS10', 'RPS11', 'RPS12', 'RPS13', 
             'RPS14', 'RPS15', 'RPS15A', 'RPS16', 'RPS17' ,'RPS18' ,'RPS19' ,'RPS20' ,'RPS21' ,'RPS23' ,'RPS24' ,'RPS25' ,'RPS26' ,'RPS27' ,
             'RPS27A', 'RPS28', 'RPS29', 'RPS30']

eIF4A_complex, eIF4A_hgnc = ['EIF4A1', 'EIF4A2'], ['HGNC:3282', 'HGNC:3284'] # helicase activity
eIF4F_complex, eIF4F_hgnc = eIF4A_complex + ['EIF4E', 'EIF4G1'], eIF4A_hgnc + ['HGNC:3287', 'HGNC:3296']
machinery_symb = {'43S preinitiation complex': ternary_complex + ['EIF1', 'EIF1AX'] + eif3_complex.loc[:, 'Approved symbol'].tolist() + forty_s_r, 
                 'cap binding, mRNA scanning': eIF4F_complex + ['EIF4B', 'EIF4H'], 
                 'loop formation': ['PABPC1'], 
                 'start codon recognition': ['EIF5', 'EIF5B']}

machinery_id = {'43S preinitiation complex': ternary_hgnc + ['HGNC:3249', 'HGNC:3250'] + eif3_complex.loc[:, 'HGNC ID (gene)'].tolist(),
                'cap binding, mRNA scanning': eIF4F_hgnc + ['HGNC:3285', 'HGNC:12741'], 
               'loop formation': ['HGNC:8554'], 
               'start codon recognition': ['HGNC:3299', 'HGNC:30793']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.pnas.org/content/98/13/7029', 
                'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3730827/',
                'https://reactome.org/PathwayBrowser/#/R-HSA-72613&SEL=R-HSA-72392&PATH=R-HSA-392499,R-HSA-72766&FLG=P08865',
                'http://ribosome.med.miyazaki-u.ac.jp/rpg.cgi?mode=orglist&org=Homo%20sapiens', 
                'https://academic.oup.com/nar/article/40/13/6199/1018198', 
                'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4231746/'] #**last citation
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'mRNA', 
          '48S: preinitiation, mRNA circularization, cap binding, mRNA scanning'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

# subunit joining - 80S ribosomal formation
energetic_cost = '1 GTP hydrolysis (eIF2-GTP)'
unit = 'ATP per n molecules'

sixty_s_r = pd.read_csv(local_data_path + 'raw/sixty_s.csv', index_col = None, skiprows = [0])

machinery_symb = {'Ribosomal subunits': forty_s_r + sixty_s_r.loc[:, 'Approved symbol'].tolist(), 
                 'eIF release': ['EIF5']}

machinery_id = {'Ribosomal subunits': sixty_s_r.loc[:, 'HGNC ID (gene)'].tolist(),
                 'eIF release': ['HGNC:3299']} 


machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['http://genesdev.cshlp.org/content/18/24/3078.full', 
                'http://ribosome.med.miyazaki-u.ac.jp/rpg.cgi?mode=orglist&org=Homo%20sapiens'] 
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'mRNA', 
          '80s ribosomal formation'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [23]:
# translation elongation

elongation_per_aa = 4
energetic_cost = L_protein*elongation_per_aa
unit = 'ATP per molecule'

# tho = pd.read_csv(local_data_path + 'raw/tho.csv', index_col = None, skiprows = [0])
# machinery_symb = {'': tho.loc[:, 'Approved symbol'].tolist()}
# machinery_id = {'': tho.loc[:, 'HGNC ID (gene)'].tolist()}

# exchange of EEF1A included as part of reaction
ribosome = sixty_s_r.loc[:, 'Approved symbol'].tolist() + forty_s_r
machinery_symb = {'Ribosome': ribosome, 
                 'Elongation Factors': ['EEF1A1', 'EEF2', 'EEF1B2', 'EIF5A']}
machinery_id = {'Ribosome': sixty_s_r.loc[:, 'HGNC ID (gene)'].tolist(), 
               'Elongation Factors': ['HGNC:3189', 'HGNC:3214', 'HGNC:3208', 'HGNC:3300']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3385960/'] 
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'mRNA', 
          'translocation_EEF1recyling'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [24]:
# # translation termination

energetic_cost = 1
unit = 'ATP per molecule'

machinery_symb = {'Release Factors': ['ETF1', 'GSPT1', 'GSPT2']}
machinery_id = {'Release Factors': ['HGNC:3477', 'HGNC:4621', 'HGNC:4622']}

machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = ['https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf']
machinery_cit = ['https://www.sciencedirect.com/science/article/pii/B9780123864970000025', 
          'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1168810/'] 
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'mRNA', 
          'termination'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [25]:
# ribosomal recycling

energetic_cost = float('nan')
unit = 'ATP per molecule'

# tho = pd.read_csv(local_data_path + 'raw/tho.csv', index_col = None, skiprows = [0])
# machinery_symb = {'': tho.loc[:, 'Approved symbol'].tolist()}
# machinery_id = {'': tho.loc[:, 'HGNC ID (gene)'].tolist()}

machinery_symb = {'General': ['ABCE1']}
machinery_id = {'General': ['HGNC:69']}


machinery_cost = cd4_prot[cd4_prot['Protein ID'].isin(sorted(set([item for sublist in list(machinery_symb.values()) for item in sublist])))]['Abundance'].sum()
machinery_cost = machinery_cost/cd4_tot

compartment = 'cytoplasm'
eng_cit = []
machinery_cit = ['https://www.sciencedirect.com/science/article/pii/B9780123864970000025'] 
comment = ''

cost.loc[('Expression', 
          'Translation', 
          'ribosome', 
          'recyling'),:] = [energetic_cost, unit, machinery_cost, machinery_symb,machinery_id, 
                                            compartment, eng_cit, machinery_cit, comment]

In [47]:
cost

Energetic Cost  \
Process    Subprocess    Substrate Reaction                                                                                                
Expression Transcription premRNA   rNTP_synthesis                                                                              3.082e+06   
                                   Transcription_Initiation                                                                           60   
                                   Capping                                                                                             4   
                         RNAP2     CTD_Phosphorylation                                                                               100   
                         preMRNA   Transcription_Elongation                                                                         1408   
                                   polyA_synthesis_transcription_termination               rNTP synthesis: 12425.0\n polyA ligation: 250   
                         RNAP      CTD_dephosphorylation                                                                             NaN   
                         preMRNA   splicing                                                                                          100   
                         intron    lariat degradation                                                                  Helicase activity   
                         mRNA      export                                                                                             10   
                                   deadenylation                                                                                     NaN   
                                   3'->5'_degradation                                       Helicase will make this a function of length   
                                   decapping_degradation                                                                                   
           Translation   aa        aa_synthesis                                                                                     7870   
                         mRNA      48S: preinitiation, mRNA circularization, cap b...  10 - Helicase activity, once per multiple tran...   
                                   80s ribosomal formation                                                   1 GTP hydrolysis (eIF2-GTP)   
                                   translocation_EEF1recyling                                                                       1574   
                                   termination                                                                                         1   
                         ribosome  recyling                                                                                          NaN   

                                                                                            Energetic Unit  \
Process    Subprocess    Substrate Reaction                                                                  
Expression Transcription premRNA   rNTP_synthesis                                         ATP per molecule   
                                   Transcription_Initiation                               ATP per molecule   
                                   Capping                                                ATP per molecule   
                         RNAP2     CTD_Phosphorylation                                    ATP per molecule   
                         preMRNA   Transcription_Elongation                               ATP per molecule   
                                   polyA_synthesis_transcription_termination              ATP per molecule   
                         RNAP      CTD_dephosphorylation                                  ATP per molecule   
                         preMRNA   splicing                                               ATP per molecule   
                         intron    lariat degradation                            

In [46]:
cost.iloc[4,]['Required Machinery Citation']

['https://www.annualreviews.org/doi/10.1146/annurev.biochem.72.121801.161551',
 'https://bionumbers.hms.harvard.edu/files/Eukaryotic%20elongation%20factors.pdf']